In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from joblib import Parallel, delayed
import warnings

warnings.filterwarnings('ignore')

from Library.ODE_timeseries import optimize_patient_parameters

def generate_patient_atlas():
    base_params = {
        'p1': 0.0152, 'd4': 2.3, 'd1': 0.0, 'd4l': 2.3,
        'k': 7.5e-9, 'tilde_n': 10.0, 'niche_capacity': 2.421e11, 
        'gamma': 6.31e-07, 'p1l': 0.0, 'p_LSCbyHSC': 0.05, 'p_HSCbyLSC': 0.95,
        'healthy_neut_per_kg': 4e8
    }

    print("1. Loading datasets...")
    # Load parameters
    params_df = pd.read_csv("IntermediateResults/venex_weighted_params.csv")
    
    # Load trajectories
    traj_path = Path("InputData/treatment_trajectories.xlsx")
    sheet_dict = pd.read_excel(traj_path, sheet_name=None)
    
    blood_all, bm_all, drug_all = None, None, None
    for sheet_name, df in sheet_dict.items():
        if 'days_lab' in df.columns or 'b_neut' in df.columns:
            blood_all = df
        elif 'days_from_bm' in df.columns or 'bm_blasts' in df.columns:
            bm_all = df
        elif 'venetoclax_start' in df.columns or 'aza_start' in df.columns:
            drug_all = df

    def standardize_id(df):
        col = 'Pseudonym' if 'Pseudonym' in df.columns else ('Patient_ID' if 'Patient_ID' in df.columns else 'ID')
        df['Patient_ID'] = df[col].astype(str)
        return df

    blood_all = standardize_id(blood_all)
    bm_all = standardize_id(bm_all)
    drug_all = standardize_id(drug_all)
    params_df['Patient_ID'] = params_df['ID'].astype(str)

    patient_ids = params_df['Patient_ID'].unique()
    optimization_jobs = []

    print(f"2. Pre-processing trajectories for {len(patient_ids)} patients...")
    for pid in patient_ids:
        p_row = params_df[params_df['Patient_ID'] == pid].iloc[0]
        phsc = float(p_row.get('pHSC_weighted', 0.95))
        coef = float(p_row.get('coef_weighted', 1.0))

        blood_df = blood_all[blood_all['Patient_ID'] == pid].sort_values('days_lab').copy()
        bm_df = bm_all[bm_all['Patient_ID'] == pid].sort_values('days_from_bm').copy()
        drug_df = drug_all[drug_all['Patient_ID'] == pid].copy()

        if blood_df.empty or bm_df.empty:
            continue

        # =====================================================================
        # 1. TREAT PRE-SCREENING (NEGATIVE DAYS) AS DAY 0 BASELINE STATE
        # =====================================================================
        blood_df.loc[blood_df['days_lab'] < 0, 'days_lab'] = 0
        bm_df.loc[bm_df['days_from_bm'] < 0, 'days_from_bm'] = 0

        # If clamping created multiple entries on Day 0, aggregate them
        blood_df = blood_df.groupby('days_lab', as_index=False)['b_neut'].mean().sort_values('days_lab')
        bm_df = bm_df.groupby('days_from_bm', as_index=False)['bm_blasts'].mean().sort_values('days_from_bm')

        # Scale neutrophils for the ODE
        blood_df.loc[blood_df['b_neut'] <= 0, 'b_neut'] = 1e-9
        blood_df['b_neut'] = blood_df['b_neut'] * 5 * 10**9 / 80

        # =====================================================================
        # 2.  DRUG SCHEDULE (EXCLUDE (0, 0) "NOT GIVEN" ENTRIES)
        # =====================================================================
        schedule = []
        for _, d_row in drug_df.iterrows():
            v_start, v_stop = d_row.get('venetoclax_start', np.nan), d_row.get('venetoclax_stop', np.nan)
            a_start, a_stop = d_row.get('aza_start', np.nan), d_row.get('aza_stop', np.nan)

            cycle_starts, cycle_stops = [], []

            # Only record active drug spans where stop > start and stop > 0
            if pd.notna(v_start) and pd.notna(v_stop) and (v_stop > v_start) and (v_stop > 0):
                cycle_starts.append(float(v_start))
                cycle_stops.append(float(v_stop))

            if pd.notna(a_start) and pd.notna(a_stop) and (a_stop > a_start) and (a_stop > 0):
                cycle_starts.append(float(a_start))
                cycle_stops.append(float(a_stop))

            if cycle_starts and cycle_stops:
                actual_start = min(cycle_starts)
                actual_stop = max(cycle_stops)
                schedule.append((actual_start, actual_stop))

        optimization_jobs.append((pid, phsc, coef, base_params.copy(), blood_df, bm_df, schedule))

    print(f"3. Launching mathematical optimizer for {len(optimization_jobs)} patients...")
    print("   (Integrating ODEs across patient-specific treatment trajectories...)")
    
    # Run optimization in parallel using all available CPU cores
    results = Parallel(n_jobs=-1)(
        delayed(optimize_patient_parameters)(
            (job[1], job[2], job[3], job[4], job[5], job[6])
        )
        for job in optimization_jobs
    )

    # Collect results
    atlas_records = []
    for job, res in zip(optimization_jobs, results):
        pid = job[0]
        if res is not None:
            atlas_records.append({
                "Patient_ID": pid,
                "Predefined_pHSC": res["Predefined_pHSC"],
                "Predefined_Coef": res["Predefined_Coef"],
                "Fitted_DrugKill_LSC": res["Fitted_DrugKill_LSC"],
                "Fitted_DrugKill_HSC": res["Fitted_DrugKill_HSC"],
                "Combined_Cost": res["RangeScaled_Combined_Error"]
            })
        else:
            print(f"   -> Optimizer failed to converge for patient {pid}")

    # 4. Save and return
    atlas_df = pd.DataFrame(atlas_records)
    Path("IntermediateResults").mkdir(parents=True, exist_ok=True)
    atlas_df.to_csv("IntermediateResults/Patient_Atlas.csv", index=False)
    
    print(f"\n Finished! Successfully optimized {len(atlas_df)}/{len(optimization_jobs)} patients.")
    print("Saved to 'IntermediateResults/Patient_Atlas.csv'.")
    return atlas_df

# Execute the pipeline
atlas_df = generate_patient_atlas()